# IEEE fraud model training: XGBoost hyperparameter tuning

Same train, validation, and test construction as `02_5_model_training_balancing_tests.ipynb`. The training set is cut once to `MAX_TRAIN_ROWS` with a stratified sample, so the fraud rate stays about 3.5%. Validation and test stay at the natural fraud rate. There is no resampling and no `scale_pos_weight`.

Optuna searches XGBoost on the columns in `TOP_40_FEATURES`. Edit that table in `src/model_training/hyperparameter_tuning.py`, or replace `FEATURE_TABLE` in the feature cell below. Each row is `[rank, feature name, gain]`. Only the feature name is trained on.

The search maximizes validation PR-AUC. Early stopping still watches validation logloss and keeps the best validation round. The test set is scored once, after the search, on the best parameters. The F1 threshold is chosen on the natural validation set.

Tracking goes to the Docker MLflow server (`http://localhost:5000`). The final model is logged to `ieee_fraud_hyperparameter_tuning`, separate from the balancing comparison.


In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "runs_entrypoints":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import mlflow
import pandas as pd

from src.model_training.hyperparameter_tuning import (
    TOP_40_FEATURES,
    feature_names,
    tune_xgboost,
)
from src.model_training.training import ModelTrainingWorkflow
from src.run_time_configuration import BaseConfigParams

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
TARGET_COLUMN = "is_fraud"
SEED = 42
MAX_TRAIN_ROWS = 100_000
N_TRIALS = 30
EARLY_STOPPING_ROUNDS = 5
EARLY_STOPPING_MIN_DELTA = 0.001
DROP_FROM_FEATURES = ["transaction_id", "seconds_from_reference", TARGET_COLUMN]
XGB_FIXED_HYPERPARAMETERS = {
    "n_estimators": 200,
    "n_jobs": 4,
}


In [ ]:
def _load_env(path: Path) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Missing {path}")
    for raw in path.read_text().splitlines():
        line = raw.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip())


_load_env(PROJECT_ROOT / ".env")

MLFLOW_TRACKING_URI = "http://localhost:5000"
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

print(f"Project root: {PROJECT_ROOT}")
print(f"Processed dir: {PROCESSED_DIR}")
print(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")


In [ ]:
df_train = pd.read_parquet(PROCESSED_DIR / "train.parquet")
df_validation = pd.read_parquet(PROCESSED_DIR / "validation.parquet")
df_test = pd.read_parquet(PROCESSED_DIR / "test.parquet")

for name, frame in {
    "train": df_train,
    "validation": df_validation,
    "test": df_test,
}.items():
    print(name, frame.shape, "pos_rate", round(float(frame[TARGET_COLUMN].mean()), 4))


In [ ]:
from sklearn.model_selection import train_test_split


def class_counts(df: pd.DataFrame) -> pd.Series:
    return df[TARGET_COLUMN].value_counts().sort_index()


def limit_training_rows(df: pd.DataFrame, max_rows: int, seed: int) -> pd.DataFrame:
    if len(df) <= max_rows:
        return df.reset_index(drop=True)
    kept, _ = train_test_split(
        df,
        train_size=max_rows,
        random_state=seed,
        stratify=df[TARGET_COLUMN],
    )
    return kept.reset_index(drop=True)


df_train_capped = limit_training_rows(df_train, MAX_TRAIN_ROWS, SEED)

print("train loaded", df_train.shape)
print(class_counts(df_train))
print("train capped", df_train_capped.shape)
print(class_counts(df_train_capped))
print("validation", df_validation.shape)
print(class_counts(df_validation))
print("test", df_test.shape)
print(class_counts(df_test))

del df_train


## Features

Edit `FEATURE_TABLE` to change the columns. Leave it as `TOP_40_FEATURES` to use the table in `src/model_training/hyperparameter_tuning.py`. That default is the gain ranking from the no-balancing run. Add, remove, or reorder rows. Each row is `[rank, feature name, gain]`. Rank and gain are notes; training uses the name only.


In [ ]:
# Edit this assignment to train on a different column list.
FEATURE_TABLE = TOP_40_FEATURES
SELECTED_FEATURES = feature_names(FEATURE_TABLE)

print(f"{len(SELECTED_FEATURES)} features")
for rank, name, gain in FEATURE_TABLE:
    print(f"{int(rank):>2}  {name}  {gain}")


## Fit the search data

No balancing. Training rows stay at the capped natural fraud rate. Validation and test are unchanged. Preprocessing keeps the selected columns and numeric values only, including native missing values.


In [ ]:
def features_and_target(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series]:
    y = df[TARGET_COLUMN]
    x = df.drop(columns=[column for column in DROP_FROM_FEATURES if column in df.columns])
    return x, y


runtime_params = BaseConfigParams(
    experiment_name="ieee_fraud_hyperparameter_tuning",
    mlflow_run_name="xgb_top40_optuna",
    training_start_date="2017-12-01",
    training_end_date="2018-03-31",
    test_start_date="2018-04-01",
    test_end_date="2018-06-30",
)
workflow = ModelTrainingWorkflow(runtime_params)
workflow.df_train = df_train_capped
workflow.df_validation = df_validation
workflow.df_test = df_test
workflow.x_train, workflow.y_train = features_and_target(df_train_capped)
workflow.x_validation, workflow.y_validation = features_and_target(df_validation)
workflow.x_test, workflow.y_test = features_and_target(df_test)
workflow.generate_preprocessed_data(
    numeric_only=True,
    selected_features=SELECTED_FEATURES,
)

print("train", workflow.x_train_preproc.shape, "pos_rate", round(float(workflow.y_train.mean()), 4))
print("validation", workflow.x_validation_preproc.shape, "pos_rate", round(float(workflow.y_validation.mean()), 4))
print("test", workflow.x_test_preproc.shape, "pos_rate", round(float(workflow.y_test.mean()), 4))
print("columns", list(workflow.x_train_preproc.columns))


## Search

Each trial trains XGBoost with early stopping on the validation set. Optuna keeps the trial with the highest validation PR-AUC. `N_TRIALS`, `EARLY_STOPPING_ROUNDS`, and `EARLY_STOPPING_MIN_DELTA` are set in the first cell.


In [ ]:
study = tune_xgboost(
    workflow,
    n_trials=N_TRIALS,
    seed=SEED,
    early_stopping_rounds=EARLY_STOPPING_ROUNDS,
    early_stopping_min_delta=EARLY_STOPPING_MIN_DELTA,
    fixed_hyperparameters=XGB_FIXED_HYPERPARAMETERS,
)

best_hyperparameters = study.best_trial.user_attrs["hyperparameters"]
print("best validation PR-AUC", round(study.best_value, 4))
print("best iteration", study.best_trial.user_attrs["best_iteration"])
print(best_hyperparameters)

trials = study.trials_dataframe().sort_values("value", ascending=False)
trials.head(10)


## Train the best parameters

One final fit, logged to MLflow. Test metrics are computed here and were not used to pick the trial. The kept model is the best validation round.


In [ ]:
results = workflow.train_model(
    hyperparameters=best_hyperparameters,
    model_type="xgboost",
    log_into_mlflow=True,
    show_plots=False,
    verbose=False,
    seed=SEED,
    run_name="xgb_top40_optuna",
    top_n_features=len(SELECTED_FEATURES),
    early_stopping_rounds=EARLY_STOPPING_ROUNDS,
    early_stopping_min_delta=EARLY_STOPPING_MIN_DELTA,
)
test_metrics = results["test_metrics"]
summary = {
    "run_id": workflow.run_time_config.mlflow_run_id,
    "n_features": len(SELECTED_FEATURES),
    "train_rows": int(len(workflow.y_train)),
    "train_pos_rate": float(workflow.y_train.mean()),
    "best_iteration": results["best_iteration"],
    "threshold": float(results["threshold"]),
    "precision": test_metrics["precision"],
    "recall": test_metrics["recall"],
    "f1": test_metrics["f1"],
    "pr_auc": test_metrics["pr_auc"],
    "auc": test_metrics["auc"],
    "ks_statistic": test_metrics["ks_statistic"],
    "optuna_val_pr_auc": study.best_value,
}
summary
